# WAAM Twin Cloud Production Workflow

This notebook is intended for **Colab** and other cloud notebook / server environments where you want functionality close to local usage, but **headless**.

Suggested flow:

1. Set up the repo, mount Google Drive (section 1b), and initialize Taichi (section 2)
2. **Edit job YAML, torch path CSV, and presets** in section 3, then run **Apply**
3. Set run controls in section 4 (steps, exports, VRAM flags)
4. Define runners once per session (sections 5–6)
5. **Launch:** section **7c** — `run_job(job, ...)`

Key variable:

- `job` — loaded from your section 3 edits (written to `jobs/notebook_session/`)

Runners:

- `run_job_live_view(...)` — short inline diagnostic frames (section 7b)
- `run_job(...)` — full batch run with manifest + optional VTK exports (section 7c)


## 1. Repository and dependency setup

Use one of these approaches:

1. clone the **direct simulator repo** into the runtime,
2. upload / unzip the direct repo,
3. mount a persistent volume / network drive.

This notebook supports both layouts:

- direct repo root: `waam_solver/` cloned into a runtime folder such as `/content/waam_twin`
- nested local-dev layout: `FYP22-01/waam_twin/`

The cells below auto-detect both.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/THEDARKDEACON/waam_solver.git"
CLONE_DIR = Path("/content/waam_twin")

if REPO_URL and not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)

candidates = [
    CLONE_DIR,
    Path("/content/waam_twin"),
    Path("/content/waam_solver"),
    Path("/content/FYP22-01/waam_twin"),
    Path("/content/FYP22-01"),
]

ROOT = next((p for p in candidates if p.exists() and (p / "requirements.txt").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate repo root. Expected a direct clone or nested FYP22-01/waam_twin layout.")

print("ROOT =", ROOT)
print("Exists:", ROOT.exists())
print("Repo URL =", REPO_URL)

ROOT = /content/waam_twin
Exists: True
Repo URL = https://github.com/THEDARKDEACON/waam_solver.git


In [2]:
# Install simulator dependencies from the detected repo root.

REQ = ROOT / "requirements.txt"
assert REQ.exists(), f"Missing requirements file: {REQ}"
%pip install -r {REQ}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.4/229.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 29.7 MB/s eta 0:00:00


In [3]:
import sys

# For the direct repo, the parent of ROOT must be on sys.path so the package
# name resolves from the cloned directory name.
PARENT = ROOT.parent
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

# Prefer CUDA when available. Set WAAM_BACKEND=cpu explicitly if you want a
# portable smoke run instead.
os.environ["WAAM_BACKEND"] = os.environ.get("WAAM_BACKEND", "cuda")
print("WAAM_BACKEND =", os.environ["WAAM_BACKEND"])
print("sys.path[0] =", sys.path[0])

WAAM_BACKEND = cuda
sys.path[0] = /content


In [4]:
# Optional: verify that CUDA is visible before Taichi initializes.
# Run this before the init cell if you want a quick environment check.

import subprocess

try:
    smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    )
    print("nvidia-smi:\n", smi)
except Exception as exc:
    print("nvidia-smi unavailable:", exc)

print("Requested WAAM_BACKEND =", os.environ.get("WAAM_BACKEND"))

nvidia-smi:
 Tesla T4, 15360, 14913, 0

Requested WAAM_BACKEND = cuda


## 1b. Google Drive persistence (Colab)

Colab VMs are ephemeral — if the runtime disconnects, `/content/cloud_runs` is lost.

Run this cell once per session to:

1. Mount Google Drive
2. Restore any prior runs from Drive into `ROOT/cloud_runs`
3. Enable automatic sync after each `run_job` / `run_job_live_view` call

With `SYNC_EACH_EXPORT = True` (default), each VTK bundle and sequence frame is copied to Drive as soon as it is written — so a disconnect mid-run still leaves partial results on Drive.

A full-folder sync still runs at the start and end of every run (including failed runs).

In [5]:
import shutil

SYNC_TO_DRIVE = True
SYNC_EACH_EXPORT = True  # copy each bundle / sequence frame to Drive as it is written
DRIVE_RUNS_SUBDIR = "waam_twin/cloud_runs"
DRIVE_MOUNTED = False
DRIVE_RUNS_ROOT = None

def ensure_drive_mounted():
    """Mount Google Drive on Colab; no-op elsewhere."""
    global DRIVE_MOUNTED, DRIVE_RUNS_ROOT
    if DRIVE_MOUNTED and DRIVE_RUNS_ROOT is not None:
        return DRIVE_RUNS_ROOT
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except ImportError:
        print("Not running in Colab — Google Drive mount skipped.")
        return None
    DRIVE_RUNS_ROOT = drive_root / DRIVE_RUNS_SUBDIR
    DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    print("DRIVE_RUNS_ROOT =", DRIVE_RUNS_ROOT)
    return DRIVE_RUNS_ROOT

def sync_subpath_to_drive(run_dir: Path, subpath: str | Path) -> Path | None:
    """Copy one file or folder under run_dir to the matching Drive path (incremental)."""
    if not globals().get("SYNC_TO_DRIVE", False):
        return None
    root = ensure_drive_mounted()
    if root is None:
        return None
    run_dir = Path(run_dir)
    local = run_dir / subpath
    if not local.exists():
        print(f"[drive] Skip sync — missing: {local}")
        return None
    dest = root / run_dir.name / subpath
    dest.parent.mkdir(parents=True, exist_ok=True)
    if local.is_dir():
        shutil.copytree(local, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(local, dest)
    return dest

def verify_drive_export(run_dir: Path, rel_path: str, *, label: str = "export") -> bool:
    """Copy one run subpath to Drive and confirm the destination exists."""
    if not globals().get("SYNC_TO_DRIVE", False):
        print(f"[drive] SKIP {label} — SYNC_TO_DRIVE is False")
        return False
    if not globals().get("SYNC_EACH_EXPORT", False):
        print(f"[drive] SKIP {label} — SYNC_EACH_EXPORT is False")
        return False
    run_dir = Path(run_dir)
    local = run_dir / rel_path
    if not local.exists():
        print(f"[drive] FAIL {label} — missing local path: {local}")
        return False
    try:
        dest = sync_subpath_to_drive(run_dir, rel_path)
        sync_subpath_to_drive(run_dir, "run_manifest.json")
    except Exception as exc:
        print(f"[drive] FAIL {label} — sync error for {rel_path}: {exc}")
        return False
    if dest is None or not Path(dest).exists():
        print(f"[drive] FAIL {label} — Drive destination missing for {rel_path}")
        return False
    dest = Path(dest)
    if dest.is_dir():
        files = [p for p in dest.rglob("*") if p.is_file()]
        vol = next(dest.glob("volume_*.vti"), None)
        extra = f", volume={vol.name}" if vol is not None else ""
        print(f"[drive] CONFIRMED {label} → {dest} ({len(files)} files{extra})")
    else:
        print(f"[drive] CONFIRMED {label} → {dest} ({dest.stat().st_size} bytes)")
    return True

def sync_run_to_drive(run_dir: Path, *, replace: bool = True) -> Path | None:
    """Copy one run folder to Google Drive (best-effort)."""
    if not globals().get("SYNC_TO_DRIVE", False):
        return None
    root = ensure_drive_mounted()
    if root is None:
        return None
    run_dir = Path(run_dir)
    if not run_dir.exists():
        print(f"[drive] Skip sync — missing run dir: {run_dir}")
        return None
    dest = root / run_dir.name
    if replace and dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest, dirs_exist_ok=not replace)
    print(f"[drive] Synced {'(full)' if replace else '(merge)'} → {dest}")
    return dest

def restore_runs_from_drive():
    """Pull any Drive runs not already present locally."""
    root = ensure_drive_mounted()
    if root is None:
        return []
    local_root = ROOT / "cloud_runs"
    local_root.mkdir(parents=True, exist_ok=True)
    restored = []
    for src in sorted(root.iterdir()):
        if not src.is_dir():
            continue
        dest = local_root / src.name
        if dest.exists():
            continue
        shutil.copytree(src, dest)
        restored.append(dest.name)
        print(f"[drive] Restored {src.name}")
    if not restored:
        print("[drive] No new runs to restore (local copy already up to date).")
    return restored

restore_runs_from_drive()


Mounted at /content/drive
DRIVE_RUNS_ROOT = /content/drive/MyDrive/waam_twin/cloud_runs
[drive] Restored live_view_run
[drive] Restored production_run
[drive] Restored widget_run


['live_view_run', 'production_run', 'widget_run']

## 2. Imports and runtime init

Run this after repository setup. It imports `waam_twin`, initializes Taichi, and defines a `reset_taichi` compatibility shim for older clones.

In [6]:
import copy
import os
import json
import math
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import yaml

from waam_twin.platform import init_taichi
try:
    from waam_twin.platform import reset_taichi
except ImportError:
    import taichi as ti
    import waam_twin.platform as _platform_mod

    def reset_taichi():
        ti.reset()
        if hasattr(_platform_mod, "_taichi_initialized"):
            _platform_mod._taichi_initialized = False
        if hasattr(_platform_mod, "_profile"):
            _platform_mod._profile = None
from waam_twin import WAAMTwin
from waam_twin.job import load_job_config

profile = init_taichi(backend=os.environ["WAAM_BACKEND"])
profile

[Taichi] version 1.7.4, llvm 15.0.4, commit b4b956fd, linux, python 3.12.13
[Taichi] Starting on arch=cuda
[waam_twin] Backend=cuda  tier=ultra  vram=15360MB  ram=12975MB


PlatformProfile(backend='cuda', device_name='cuda', vram_mb=15360, ram_mb=12975, tier='ultra')

## 3. Job configuration (edit before launch)

Edit the three cells below **in place**, then run **3d Apply**.

Files are written to `jobs/notebook_session/` and loaded into `job` for all run cells.

- **3a** — job YAML (`process`, `deposition`, `simulation.preset`, heat source, etc.)
- **3b** — torch travel path CSV (`x_mm,y_mm,z_mm` waypoints)
- **3c** — grid fidelity presets (`minimal` … `ultra`); changing these re-initializes Taichi

Re-run **3d Apply** after any edit before launching a simulation.


In [23]:
# 3a — Job YAML
JOB_YAML = """\
simulation:
  preset: ultra
  backend: cuda
  enable_vof: true
  enable_csf_tension: true
  enable_wetting: true
  enable_hydrostatic_gravity: true
  enable_bead_freeze: true
  enable_recoil: true
  use_recoil_clausius_clapeyron: true #true for CCCC, false for RR
  enable_lorentz: true
  enable_gas_shear: true
  enable_droplet_impact_pressure: true
  enable_enthalpy_cap: true
  arc_surface_weighting: true

material: materials/validated/ER70S-6.v1.yaml
calibration: null

# Goldak (1984) moving double-ellipsoid — asymmetric pool for travel along +x
heat_source: goldak
goldak:
  ff: 0.6
  fr: 0.4
  depth_front_mm: 0.9
  depth_rear_mm: 1.8

# Arc penetration attenuation in solid (δ ≈ 2 mm); vaporization enthalpy ceiling
arc_physics:
  penetration_mm: 2.0
  T_vapor_cap_K: 3200.0
  surface_weighting: true

advanced_physics:
  gas_jet_velocity_m_s: 12.0
  gas_shear_coeff: 1.0
  sigma_liquid_Sm: 1.0e6
  sigma_solid_Sm: 1.0e5
  lorentz_jacobi_iters: 40
  T_boiling_K: 3100.0
  L_vapor_J_kg: 6.0e6
  R_spec_vapor_J_kgK: 450.0

process:
  current_A: 100
  voltage_V: 12
  arc_efficiency: 0.70
  travel_speed_mm_s: 20 
  wire_feed_m_min: 0.1
  wire_diameter_mm: 1.2
  droplet_length_mm: 3.0
  T_ambient_K: 293
  stickout_mm: 12.0
  ctwd_mm: 15.0
  transfer_mode: auto   # globular | spray | pulsed | auto
  droplet_freq_hz: 44.0
  pulse_frequency_hz: 90.0
  droplet_size_jitter: 0.12
  impact_lead_angle_deg: 8.0


deposition:
  trailing_solidify_lookback_mm: 2.5
  trailing_solidify_temp_margin_K: 35.0
  superheat_K: 500.0
  footprint_sigma_scale: 1.0

surface_wetting:
  contact_angle_deg: 80.0

heat_loss:
  convection: true
  radiation: true
  h_conv: 35.0

reference:
  pool_width_mm: 7.0
  pool_depth_mm: 2.8
  source: macrograph_ER70S-6_bead_on_plate

# Achievable pool envelope on 88×44×44 @ dx=0.3 mm (Goldak + surface-weighted arc).
# Macro reference above is experimental; use model_reference for CI calibration checks.
model_reference:
  grid: [88, 44, 44]
  dx_mm: 0.3
  n_steps: 2000
  pool_width_mm: 4.2
  pool_depth_mm: 5.4

probes:
  - name: substrate_behind_arc
    x_mm: 8.0
    y_mm: 20.0
    z_mm: 1.5
"""


In [24]:
# 3b — Torch path CSV
TORCH_PATH_CSV = """\
x_mm,y_mm,z_mm
5,10,0
35,10,0
35,30,0
5,30,0
5,10,0
"""


In [25]:
# 3c — Fidelity presets (config/presets.yaml)
PRESETS_YAML = """\
# waam_twin v2 fidelity presets — grid scales with host VRAM via platform.auto_grid()

minimal:
  vram_budget_mb: 512
  domain_mm: [40, 20, 15]
  target_dx_mm: 0.5
  max_tracers: 5000
  use_srt: true

standard:
  vram_budget_mb: 1536
  domain_mm: [80, 40, 25]
  target_dx_mm: 0.3
  max_tracers: 20000
  use_srt: true

high:
  vram_budget_mb: 8192
  domain_mm: [120, 60, 40]
  target_dx_mm: 0.05
  max_tracers: 50000
  use_srt: false

ultra:
  vram_budget_mb: 14000
  domain_mm: [200, 100, 60]
  target_dx_mm: 0.05
  max_tracers: 100000
  use_srt: false
"""


In [26]:
# 3d — Apply notebook edits → jobs/notebook_session/ + load `job`

import waam_twin.platform as platform_mod
from waam_twin.job import parse_torch_path


def write_job_yaml(job_dict: dict, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(job_dict, sort_keys=False))
    return out_path


def apply_job_config():
    session = ROOT / "jobs" / "notebook_session"
    session.mkdir(parents=True, exist_ok=True)

    job_path = session / "job.yaml"
    csv_path = session / "torch_path.csv"
    presets_path = session / "presets.yaml"

    job_path.write_text(JOB_YAML.strip() + "\n")
    csv_path.write_text(TORCH_PATH_CSV.strip() + "\n")
    presets_path.write_text(PRESETS_YAML.strip() + "\n")

    platform_mod._PRESETS_PATH = presets_path

    loaded = load_job_config(job_path)
    loaded["torch_path_csv"] = csv_path.relative_to(ROOT).as_posix()

    waypoints = parse_torch_path(loaded)
    presets = platform_mod.load_presets()
    preset_name = str(loaded.get("simulation", {}).get("preset", "standard"))

    # Preset edits affect grid sizing — re-init Taichi when session is already warm.
    if globals().get("profile") is not None:
        reset_taichi()
        globals()["profile"] = init_taichi(backend=os.environ["WAAM_BACKEND"])

    print("Applied notebook job configuration")
    print("  job file:", job_path)
    print("  torch CSV:", csv_path, f"({len(waypoints)} waypoints)")
    print("  presets:", presets_path, "→", list(presets.keys()))
    print("  simulation.preset:", preset_name, "(override with PRESET_OVERRIDE in section 4)")
    print("  process.current_A:", loaded.get("process", {}).get("current_A"))
    print("  process.travel_speed_mm_s:", loaded.get("process", {}).get("travel_speed_mm_s"))
    return loaded


job = apply_job_config()
job


[Taichi] Starting on arch=cuda
[waam_twin] Backend=cuda  tier=ultra  vram=15360MB  ram=12975MB
Applied notebook job configuration
  job file: /content/waam_twin/jobs/notebook_session/job.yaml
  torch CSV: /content/waam_twin/jobs/notebook_session/torch_path.csv (5 waypoints)
  presets: /content/waam_twin/jobs/notebook_session/presets.yaml → ['minimal', 'standard', 'high', 'ultra']
  simulation.preset: ultra (override with PRESET_OVERRIDE in section 4)
  process.current_A: 100
  process.travel_speed_mm_s: 20


{'simulation': {'preset': 'ultra',
  'backend': 'cuda',
  'enable_vof': True,
  'enable_csf_tension': True,
  'enable_wetting': True,
  'enable_hydrostatic_gravity': True,
  'enable_bead_freeze': True,
  'enable_recoil': True,
  'use_recoil_clausius_clapeyron': True,
  'enable_lorentz': True,
  'enable_gas_shear': True,
  'enable_droplet_impact_pressure': True,
  'enable_enthalpy_cap': True,
  'arc_surface_weighting': True},
 'material': 'materials/validated/ER70S-6.v1.yaml',
 'calibration': None,
 'heat_source': 'goldak',
 'goldak': {'ff': 0.6, 'fr': 0.4, 'depth_front_mm': 0.9, 'depth_rear_mm': 1.8},
 'arc_physics': {'penetration_mm': 2.0,
  'T_vapor_cap_K': 3200.0,
  'surface_weighting': True},
 'advanced_physics': {'gas_jet_velocity_m_s': 12.0,
  'gas_shear_coeff': 1.0,
  'sigma_liquid_Sm': '1.0e6',
  'sigma_solid_Sm': '1.0e5',
  'lorentz_jacobi_iters': 40,
  'T_boiling_K': 3100.0,
  'L_vapor_J_kg': '6.0e6',
  'R_spec_vapor_J_kgK': 450.0},
 'process': {'current_A': 100,
  'voltage_V

## 4. Run configuration

Notebook-side controls for each launch (section 3 holds the job / path / grid definitions):

- `PRESET_OVERRIDE`: optional fidelity override (`None` uses `simulation.preset` from section 3a)
- `N_STEPS`: simulation duration (`3500+` for meaningful pool/bead metrics)
- `EXPORT_BUNDLE`, `EXPORT_SEQUENCE`, `SEQUENCE_EVERY`: output controls
- `RESET_TAICHI_EACH_RUN`, `FREE_VRAM_AFTER_RUN`, `KEEP_TWIN_IN_MEMORY`: VRAM management

- `LOG_PROGRESS_EVERY`: heartbeat print every N simulation steps so long runs do not look stuck

Tip: keep `EXPORT_SEQUENCE = False` on the first full run if you want the fastest turnaround. When enabled, frames are exported during the main run, `sequence.pvd` is rewritten after each frame, and each export prints `[drive] CONFIRMED ...` after a verified Drive copy.


In [27]:
PRESET_OVERRIDE = None   # None | "minimal" | "standard" | "high" | "ultra"
N_STEPS = 250000
EXPORT_BUNDLE = True
EXPORT_SEQUENCE = True
SEQUENCE_EVERY = 500
LOG_PROGRESS_EVERY = 500   # print heartbeat every N sim steps (0 = off)
RESET_TAICHI_EACH_RUN = True
FREE_VRAM_AFTER_RUN = True
KEEP_TWIN_IN_MEMORY = False
# SYNC_TO_DRIVE is set in section 1b; runs auto-copy to Google Drive after each job.

print({
    "PRESET_OVERRIDE": PRESET_OVERRIDE,
    "N_STEPS": N_STEPS,
    "EXPORT_BUNDLE": EXPORT_BUNDLE,
    "EXPORT_SEQUENCE": EXPORT_SEQUENCE,
    "SEQUENCE_EVERY": SEQUENCE_EVERY,
    "LOG_PROGRESS_EVERY": LOG_PROGRESS_EVERY,
    "RESET_TAICHI_EACH_RUN": RESET_TAICHI_EACH_RUN,
    "FREE_VRAM_AFTER_RUN": FREE_VRAM_AFTER_RUN,
    "KEEP_TWIN_IN_MEMORY": KEEP_TWIN_IN_MEMORY,
    "SYNC_TO_DRIVE": globals().get("SYNC_TO_DRIVE", False),
    "SYNC_EACH_EXPORT": globals().get("SYNC_EACH_EXPORT", False),
})


{'PRESET_OVERRIDE': None, 'N_STEPS': 250000, 'EXPORT_BUNDLE': True, 'EXPORT_SEQUENCE': True, 'SEQUENCE_EVERY': 100, 'RESET_TAICHI_EACH_RUN': True, 'FREE_VRAM_AFTER_RUN': True, 'KEEP_TWIN_IN_MEMORY': False, 'SYNC_TO_DRIVE': True, 'SYNC_EACH_EXPORT': True}


## 5. Production-style batch runner

Define `run_job(...)` once per session. It:

- builds the twin from a job
- runs the simulation without inline image display
- collects telemetry and writes a manifest with run status
- optionally exports bundle / sequence VTK outputs

Use this for clean batch runs, sweeps, and production-style exports.

In [28]:
def run_job(
    job_dict: dict,
    *,
    preset_override: str | None = None,
    n_steps: int = 1200,
    export_bundle: bool = False,
    export_sequence: bool = False,
    sequence_every: int = 100,
    out_dir: Path | None = None,
    tag: str | None = None,
):
    from waam_twin.export import write_pvd
    from waam_twin.job import load_job_config, parse_torch_path
    from waam_twin.torch_path import TorchPathDriver, clamp_torch_to_domain

    work_dir = ROOT / "jobs" / "generated"
    work_dir.mkdir(parents=True, exist_ok=True)
    run_root = (out_dir or ROOT / "cloud_runs")
    run_root.mkdir(parents=True, exist_ok=True)

    tag = tag or f"run_{int(time.time())}"
    run_dir = run_root / tag
    run_dir.mkdir(parents=True, exist_ok=True)

    job_file = run_dir / f"{tag}.yaml"
    write_job_yaml(job_dict, job_file)

    manifest_path = run_dir / "run_manifest.json"
    paths = {
        "run_dir": str(run_dir),
        "job_file": str(job_file),
        "manifest": str(manifest_path),
    }
    checks = {
        "requested_steps": int(n_steps),
        "bundle_ok": not export_bundle,
        "sequence_ok": not export_sequence,
    }
    warnings = []
    telem = {}
    twin = None
    elapsed = None

    def write_manifest(status: str, error: str | None = None, tb: str | None = None):
        manifest = {
            "tag": tag,
            "status": status,
            "completed": status.startswith("completed"),
            "elapsed_s": None if elapsed is None else round(elapsed, 3),
            "preset_override": preset_override,
            "n_steps": int(n_steps),
            "export_bundle": export_bundle,
            "export_sequence": export_sequence,
            "sequence_every": sequence_every,
            "job_file": str(job_file),
            "telemetry": telem,
            "paths": paths,
            "checks": checks,
            "warnings": warnings,
            "error": error,
            "traceback": tb,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2))
        return manifest

    def drive_sync_export(rel_path: str, *, label: str = "export") -> bool:
        return verify_drive_export(run_dir, rel_path, label=label)

    write_manifest("running")
    print(f"[run] tag={tag}")
    print(f"[run] local run_dir={run_dir}")
    print(f"[run] exports: bundle={export_bundle} sequence={export_sequence} sequence_every={sequence_every}")
    print(
        f"[run] drive: SYNC_TO_DRIVE={globals().get('SYNC_TO_DRIVE', False)} "
        f"SYNC_EACH_EXPORT={globals().get('SYNC_EACH_EXPORT', False)}"
    )
    try:
        sync_run_to_drive(run_dir, replace=False)
    except Exception as exc:
        print(f"[drive] Early sync failed: {exc}")
    keep_twin = bool(globals().get("KEEP_TWIN_IN_MEMORY", False))
    reset_each_run = bool(globals().get("RESET_TAICHI_EACH_RUN", False))
    free_vram_after_run = bool(globals().get("FREE_VRAM_AFTER_RUN", False))
    if reset_each_run:
        reset_taichi()
        init_taichi(backend=os.environ["WAAM_BACKEND"])
    t0 = time.perf_counter()
    try:
        twin = WAAMTwin.from_job(job_file, preset_override=preset_override)
        twin.reset()
        job_cfg = load_job_config(job_file)
        g = twin.grid

        waypoints = parse_torch_path(job_cfg)
        if not waypoints:
            cy = (g.ny // 2) * g.dx
            waypoints = [(0.01, cy, 0.0)]
        driver = TorchPathDriver(waypoints, twin.travel_speed_m_s)

        interpass = int(getattr(twin, "_interpass_cooling_steps", 0) or 0)
        prev_seg = -1
        vti_paths = []
        frame_steps = []
        frame_idx = 0

        if export_sequence:
            seq_dir = run_dir / "sequence"
            every_n = max(1, int(sequence_every))
            frame_steps = list(range(0, int(n_steps), every_n))
            if not frame_steps or frame_steps[-1] != int(n_steps) - 1:
                frame_steps.append(int(n_steps) - 1)
            checks["expected_sequence_frames"] = len(frame_steps)
            paths["sequence_dir"] = str(seq_dir)

            def rewrite_sequence_pvd():
                if not vti_paths:
                    return
                pvd_path = seq_dir / "sequence.pvd"
                write_pvd(pvd_path, vti_paths)
                paths["sequence_pvd"] = str(pvd_path)
                drive_sync_export("sequence/sequence.pvd", label=f"pvd ({len(vti_paths)} frames)")

        progress_every = int(globals().get("LOG_PROGRESS_EVERY", 500) or 0)
        loop_t0 = time.perf_counter()
        if export_sequence:
            print(
                f"[run] sequence plan: {len(frame_steps)} frames, first at step {frame_steps[0]}, "
                f"next every {max(1, int(sequence_every))} steps"
            )
        print(f"[run] starting simulation loop: {n_steps} steps")

        for step, x, y, z in driver.positions_for_steps(n_steps, g.dt):
            seg = driver.segment_index_at_distance(driver.distance_at_step(step, g.dt))
            if interpass > 0 and seg > prev_seg and prev_seg >= 0:
                park = driver.segment_end(prev_seg)
                px, py, pz = park if park else (x, y, z)
                for _ in range(interpass):
                    cx, cy = clamp_torch_to_domain(px, py, g.nx, g.ny, g.dx)
                    twin.step(cx, cy, is_welding=False, torch_z_m=pz)
            prev_seg = seg

            cx, cy = clamp_torch_to_domain(x, y, g.nx, g.ny, g.dx)
            twin.step(cx, cy, is_welding=True, torch_z_m=z)

            if step == 0:
                sim_time_s = n_steps * g.dt
                print(
                    f"[run] step 1/{n_steps} ok | dt={g.dt*1e6:.1f}µs | planned sim time ~{sim_time_s:.2f}s"
                )
            elif progress_every > 0 and (step + 1) % progress_every == 0:
                pct = 100.0 * (step + 1) / n_steps
                elapsed_loop_s = time.perf_counter() - loop_t0
                frames_done = checks.get("sequence_frames_exported", 0)
                print(
                    f"[run] step {step + 1}/{n_steps} ({pct:.1f}%) | "
                    f"elapsed={elapsed_loop_s:.0f}s | twin_step={twin._step_n} | frames_on_drive={frames_done}"
                )

            if export_sequence and frame_idx < len(frame_steps) and step == frame_steps[frame_idx]:
                step_tag = f"step_{twin._step_n:06d}"
                frame_dir = seq_dir / f"frame_{frame_idx:04d}"
                rel_frame = frame_dir.relative_to(run_dir).as_posix()
                print(
                    f"[export] frame {frame_idx + 1}/{len(frame_steps)} at sim step {step} "
                    f"(twin step {twin._step_n}) → {rel_frame}"
                )
                frame_paths = twin.export_research_bundle(
                    frame_dir,
                    tag=step_tag,
                    include_surface=True,
                    include_tracers=(frame_idx == 0 or frame_idx % 5 == 0),
                )
                if "volume" in frame_paths:
                    vti_paths.append(frame_paths["volume"])
                checks["sequence_frames_exported"] = frame_idx + 1
                write_manifest("running")
                ok = drive_sync_export(rel_frame, label=f"frame {frame_idx + 1}/{len(frame_steps)}")
                if not ok:
                    warnings.append(f"Drive export not confirmed for {rel_frame}")
                rewrite_sequence_pvd()
                frame_idx += 1

        telem = twin.get_telemetry()
        elapsed = time.perf_counter() - t0

        if export_bundle:
            bundle_dir = run_dir / "bundle"
            paths.update(twin.export_research_bundle(bundle_dir, tag=tag))
            volume_path = paths.get("volume")
            checks["bundle_ok"] = bool(volume_path and Path(volume_path).exists())
            if not checks["bundle_ok"]:
                warnings.append("Bundle export requested but no volume VTI was found.")
            write_manifest("running")
            drive_sync_export("bundle", label="final bundle")

        if export_sequence:
            paths["sequence_vti_count"] = len(vti_paths)
            checks["sequence_ok"] = len(vti_paths) >= checks.get("expected_sequence_frames", 0)
            rewrite_sequence_pvd()
            if not checks["sequence_ok"]:
                warnings.append(
                    f"Sequence export requested {checks.get('expected_sequence_frames', 0)} frames but wrote {len(vti_paths)}."
                )

        mass_ratio = telem.get("mass_balance_ratio")
        checks["mass_balance_ratio_finite"] = bool(np.isfinite(mass_ratio)) if mass_ratio is not None else False
        if not checks["mass_balance_ratio_finite"]:
            warnings.append("mass_balance_ratio is missing or non-finite.")

        status = "completed_with_warnings" if warnings else "completed"
        manifest = write_manifest(status)
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        manifest = write_manifest("failed", error=repr(exc), tb=traceback.format_exc())
        raise RuntimeError(f"Run '{tag}' failed. See {manifest_path}") from exc
    finally:
        try:
            sync_run_to_drive(run_dir)
        except Exception as exc:
            print(f"[drive] Final sync failed: {exc}")
        if free_vram_after_run and not keep_twin:
            twin = None
            reset_taichi()
            init_taichi(backend=os.environ["WAAM_BACKEND"])

    return {
        "status": manifest["status"],
        "job_file": str(job_file),
        "elapsed_s": round(elapsed, 3),
        "telemetry": telem,
        "paths": paths,
        "checks": checks,
        "warnings": warnings,
        "manifest": manifest,
        "twin": twin if keep_twin else None,
        "drive_dir": str((ensure_drive_mounted() or Path(".")) / run_dir.name) if globals().get("SYNC_TO_DRIVE") and ensure_drive_mounted() else None,
    }

## 6. Live-view runner

Define `run_job_live_view(...)` once per session. It:

- creates the twin and runs the actual simulation path
- displays diagnostic frame images inline while the run progresses
- saves those frames to disk for later inspection

Use this when you want to watch the run develop and catch bad bead shape or pool behavior early.

In [29]:
from IPython.display import Image, display, clear_output


def run_job_live_view(
    job_dict: dict,
    *,
    preset_override: str | None = None,
    n_steps: int = 1200,
    display_every: int = 150,
    max_frames: int = 12,
    field: str = "Temperature_K",
    cmap: str = "inferno",
    out_dir: Path | None = None,
    tag: str | None = None,
    clear_between_frames: bool = False,
):
    from waam_twin.job import load_job_config, parse_torch_path
    from waam_twin.torch_path import TorchPathDriver, clamp_torch_to_domain

    def _write_job_yaml_local(job_dict: dict, out_path: Path):
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(yaml.safe_dump(job_dict, sort_keys=False))
        return out_path

    def _step_twin_compat(twin, x_m: float, y_m: float, is_welding: bool, z_m: float | None = None):
        try:
            if z_m is not None:
                twin.step(x_m, y_m, is_welding=is_welding, torch_z_m=z_m)
            else:
                twin.step(x_m, y_m, is_welding=is_welding)
        except TypeError:
            twin.step(x_m, y_m, is_welding=is_welding)

    def _save_slice_png_local(twin, out_path: Path, field: str = "Temperature_K", cmap: str = "inferno"):
        g = twin.grid
        if field == "Temperature_K":
            arr = g.T.to_numpy()
        elif field == "Liquid_Fraction":
            arr = g.f_l.to_numpy()
        elif field == "Speed_ms":
            ux = g.ux.to_numpy()
            uy = g.uy.to_numpy()
            uz = g.uz.to_numpy()
            arr = np.sqrt(ux * ux + uy * uy + uz * uz) * g.dx / g.dt
        else:
            raise KeyError(f"Unsupported preview field: {field}")

        mid_y = arr.shape[1] // 2
        sl = arr[:, mid_y, :].T
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.figure(figsize=(10, 4))
        plt.imshow(sl, origin="lower", aspect="auto", cmap=cmap)
        plt.colorbar(label=field)
        plt.title(f"{field} at step {twin._step_n}")
        plt.xlabel("x cell")
        plt.ylabel("z cell")
        plt.tight_layout()
        plt.savefig(out_path, dpi=150)
        plt.close()
        return out_path

    keep_twin = bool(globals().get("KEEP_TWIN_IN_MEMORY", False))
    reset_each_run = bool(globals().get("RESET_TAICHI_EACH_RUN", False))
    free_vram_after_run = bool(globals().get("FREE_VRAM_AFTER_RUN", False))

    run_root = (out_dir or ROOT / "cloud_runs")
    run_root.mkdir(parents=True, exist_ok=True)
    tag = tag or f"live_view_{int(time.time())}"
    run_dir = run_root / tag
    run_dir.mkdir(parents=True, exist_ok=True)

    job_file = run_dir / f"{tag}.yaml"
    _write_job_yaml_local(job_dict, job_file)

    if reset_each_run:
        reset_taichi()
        init_taichi(backend=os.environ["WAAM_BACKEND"])

    twin = WAAMTwin.from_job(job_file, preset_override=preset_override)
    twin.reset()
    job_cfg = load_job_config(job_file)
    g = twin.grid

    waypoints = parse_torch_path(job_cfg)
    if not waypoints:
        cy = (g.ny // 2) * g.dx
        waypoints = [(0.01, cy, 0.0)]
    driver = TorchPathDriver(waypoints, twin.travel_speed_m_s)

    interpass = int(getattr(twin, "_interpass_cooling_steps", 0) or 0)
    prev_seg = -1
    frame_dir = run_dir / "live_frames"
    rows = []
    frame_idx = 0

    for step, x, y, z in driver.positions_for_steps(n_steps, g.dt):
        seg = driver.segment_index_at_distance(driver.distance_at_step(step, g.dt))
        if interpass > 0 and seg > prev_seg and prev_seg >= 0:
            park = driver.segment_end(prev_seg)
            px, py, pz = park if park else (x, y, z)
            for _ in range(interpass):
                cx, cy = clamp_torch_to_domain(px, py, g.nx, g.ny, g.dx)
                _step_twin_compat(twin, cx, cy, is_welding=False, z_m=pz)
        prev_seg = seg

        cx, cy = clamp_torch_to_domain(x, y, g.nx, g.ny, g.dx)
        _step_twin_compat(twin, cx, cy, is_welding=True, z_m=z)

        should_show = (twin._step_n % display_every == 0) or (step == n_steps - 1)
        if not should_show:
            continue

        png_path = frame_dir / f"frame_{frame_idx:04d}_step_{twin._step_n:06d}.png"
        _save_slice_png_local(twin, png_path, field=field, cmap=cmap)
        telem = twin.get_telemetry()
        row = {
            "frame": frame_idx,
            "step": twin._step_n,
            "sim_time_ms": telem.get("sim_time_ms"),
            "pool_width_mm": telem.get("pool_width_mm"),
            "pool_depth_mm": telem.get("pool_depth_mm"),
            "bead_height_mm": telem.get("bead_height_mm"),
            "mass_balance_ratio": telem.get("mass_balance_ratio"),
            "png": str(png_path),
        }
        rows.append(row)

        if clear_between_frames:
            clear_output(wait=True)
        print(row)
        display(Image(filename=str(png_path)))

        frame_idx += 1
        if frame_idx >= max_frames:
            break

    live_df = pd.DataFrame(rows)
    live_csv = run_dir / "live_view_manifest.csv"
    live_df.to_csv(live_csv, index=False)

    try:
        sync_run_to_drive(run_dir)
    except Exception as exc:
        print(f"[drive] Live-view sync failed: {exc}")

    if free_vram_after_run and not keep_twin:
        twin = None
        reset_taichi()
        init_taichi(backend=os.environ["WAAM_BACKEND"])

    return {
        "run_dir": str(run_dir),
        "job_file": str(job_file),
        "frame_dir": str(frame_dir),
        "live_view_manifest_csv": str(live_csv),
        "live_df": live_df,
        "twin": twin if keep_twin else None,
    }

## 7. Run simulations

Pick one or more execution paths below.

- Section **3d Apply** must be run first so `job` is loaded.
- Section **4** controls preset, step count, and exports.
- Production runs pass `job` from section 3 into `run_job(...)`.

### 7b. Live view (inline diagnostics)

use **7c** instead

In [ ]:
live_view_result = run_job_live_view(
    job,
    preset_override=PRESET_OVERRIDE,
    n_steps=min(N_STEPS, 1200),
    display_every=150,
    max_frames=8,
    field="Temperature_K",
    tag="live_view_run",
    clear_between_frames=False,
)
live_view_result["live_df"]

### 7c. Full production batch run (recommended)

Runs the current `job` from section 3.
This is the main cell for a full weld + deposition run.

In [30]:
RUN_TAG = f"production_run_{int(time.time())}"
expected_frames = (N_STEPS // max(SEQUENCE_EVERY, 1)) + 1
print("[preflight] production run")
print("  RUN_TAG =", RUN_TAG)
print("  local dir =", ROOT / "cloud_runs" / RUN_TAG)
print("  drive dir =", (globals().get("DRIVE_RUNS_ROOT") or Path("MyDrive/waam_twin/cloud_runs")) / RUN_TAG)
print("  N_STEPS =", N_STEPS, "| EXPORT_SEQUENCE =", EXPORT_SEQUENCE, "| SEQUENCE_EVERY =", SEQUENCE_EVERY)
print("  expected sequence frames ~", expected_frames)
print("  SYNC_TO_DRIVE =", globals().get("SYNC_TO_DRIVE"), "| SYNC_EACH_EXPORT =", globals().get("SYNC_EACH_EXPORT"))
print("  During the run you should see [export] frame ... then [drive] CONFIRMED frame ...")
print("  If not, re-run section 5 (run_job definition) and section 1b (Drive mount).")

run_result = run_job(
    job,
    preset_override=PRESET_OVERRIDE,
    n_steps=N_STEPS,
    export_bundle=EXPORT_BUNDLE,
    export_sequence=EXPORT_SEQUENCE,
    sequence_every=SEQUENCE_EVERY,
    tag=RUN_TAG,
)
print("[done] status =", run_result["status"])
print("[done] frames exported =", run_result["checks"].get("sequence_frames_exported"))
print("[done] drive_dir =", run_result.get("drive_dir"))
run_result["telemetry"]


[drive] Synced (full) → /content/drive/MyDrive/waam_twin/cloud_runs/production_run
[Taichi] Starting on arch=cuda
[waam_twin] Backend=cuda  tier=ultra  vram=15360MB  ram=12975MB
[WAAMGrid] Domain: 650×325×195  |  dx=0.308mm  dt=30.764µs  τ=0.5009  τ_T=0.5061
[WAAMTwin] Material: ER70S-6  status=calibrated
[WAAMTwin] T-dependent tables: cp, k, dγ/dT, μ→τ on GPU
[WAAMTwin] Heat source: goldak
[WAAMTwin] VRAM estimate: 11630.0 MB
[WAAMTwin] α_lu=0.002048  τ_T=0.5061
[WAAMTwin] Collision: MRT
[WAAMTwin] Preset=ultra  grid=650×325×195  dx=0.308mm
[WAAMTwin] Grid reset. T_ambient=293.0K  test_fluid=False
[drive] Synced (full) → /content/drive/MyDrive/waam_twin/cloud_runs/production_run
[Taichi] Starting on arch=cuda
[waam_twin] Backend=cuda  tier=ultra  vram=15360MB  ram=12975MB


KeyboardInterrupt: 

### 7d. Batch parameter sweeps

Compare multiple job variants in one session — useful for production-style tuning on cloud GPUs.

In [ ]:
def _sweep_job(base: dict, overrides: dict) -> dict:
    out = copy.deepcopy(base)
    for key, value in overrides.items():
        cur = out
        parts = key.split(".")
        for part in parts[:-1]:
            cur = cur.setdefault(part, {})
        cur[parts[-1]] = value
    return out


def flatten_result(label: str, result: dict, overrides: dict) -> dict:
    row = {
        "label": label,
        "elapsed_s": result["elapsed_s"],
        **overrides,
    }
    row.update(result["telemetry"])
    return row


cases = [
    ("globular_5mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 5.0,
        "process.transfer_mode": "globular",
    }),
    ("spray_8mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 8.0,
        "process.transfer_mode": "spray",
    }),
    ("spray_12mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 12.0,
        "process.transfer_mode": "spray",
    }),
]

rows = []
for label, ov in cases:
    job_case = _sweep_job(job, ov)
    res = run_job(job_case, n_steps=1000, tag=label)
    rows.append(flatten_result(label, res, ov))

df = pd.DataFrame(rows)
df[[
    "label",
    "process.travel_speed_mm_s",
    "process.transfer_mode",
    "pool_depth_mm",
    "bead_height_mm",
    "mass_balance_ratio",
    "droplet_transfer_mode",
    "droplet_impact_velocity_ms",
    "elapsed_s",
]]

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["label"], df["bead_height_mm"], marker="o", label="bead_height_mm")
plt.plot(df["label"], df["pool_depth_mm"], marker="s", label="pool_depth_mm")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df["label"], df["mass_balance_ratio"], marker="o")
plt.axhline(1.0, color="red", linestyle="--", label="ideal")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

### 7e. Rank sweep results

Sort candidate jobs with a simple score instead of eyeballing every run.

In [ ]:
# Adjust these targets for your study.
TARGETS = {
    "bead_height_mm": 1.2,
    "pool_depth_mm": 2.8,
    "mass_balance_ratio": 1.0,
}
WEIGHTS = {
    "bead_height_mm": 1.0,
    "pool_depth_mm": 1.0,
    "mass_balance_ratio": 1.5,
}

rank_df = df.copy()
score = np.zeros(len(rank_df), dtype=float)
for key, target in TARGETS.items():
    if key not in rank_df:
        continue
    err = np.abs(rank_df[key] - target) / max(abs(target), 1e-9)
    score += WEIGHTS.get(key, 1.0) * err
rank_df["score"] = score
rank_df = rank_df.sort_values("score").reset_index(drop=True)
rank_df[[
    "label",
    "score",
    "bead_height_mm",
    "pool_depth_mm",
    "mass_balance_ratio",
    "droplet_transfer_mode",
    "droplet_impact_velocity_ms",
    "elapsed_s",
]]

## 8. Post-processing

### 8a. Export time sequence for ParaView

Closest cloud equivalent to a local inspection workflow. Enable `EXPORT_SEQUENCE` in section 4, or run this dedicated sequence export.

In [ ]:
sequence_result = run_job(
    job,
    preset_override=PRESET_OVERRIDE,
    n_steps=N_STEPS,
    export_bundle=False,
    export_sequence=True,
    sequence_every=SEQUENCE_EVERY,
    tag=f"sequence_only_{int(time.time())}",
)
sequence_result["paths"]

## 8b. Inspect bundle / VTK arrays in-notebook

Uses `run_result` from **7c** — whichever you ran most recently.

In [ ]:
# Inspect the most recent batch run that exported a bundle (7c).
inspect_result = globals().get("run_result") or globals().get("result")
if inspect_result is None:
    raise RuntimeError("Run section 7c first to produce a VTK bundle.")

bundle = inspect_result["paths"]
vti_path = Path(bundle["volume"])
grid = pv.read(str(vti_path))

print("VTI:", vti_path)
print("n_cells:", grid.n_cells)
print("spacing:", grid.spacing)
print("arrays:", sorted(grid.cell_data.keys()))

for name in ["Temperature_K", "Liquid_Fraction", "Speed_ms", "Curvature_kappa", "BodyForce_Z_ms2"]:
    if name in grid.cell_data:
        arr = np.asarray(grid.cell_data[name])
        print(name, "min=", np.nanmin(arr), "max=", np.nanmax(arr), "mean=", np.nanmean(arr))


In [ ]:
def show_mid_y_slice(grid, field="Temperature_K", cmap="inferno"):
    dims = tuple(int(v - 1) for v in grid.dimensions)
    arr = np.asarray(grid.cell_data[field]).reshape(dims, order="F")
    mid_y = arr.shape[1] // 2
    sl = arr[:, mid_y, :].T
    plt.figure(figsize=(11, 4))
    plt.imshow(sl, origin="lower", aspect="auto", cmap=cmap)
    plt.colorbar(label=field)
    plt.title(f"Mid-Y slice: {field}")
    plt.xlabel("x cell")
    plt.ylabel("z cell")
    plt.tight_layout()
    plt.show()


show_mid_y_slice(grid, "Temperature_K")
show_mid_y_slice(grid, "Liquid_Fraction", cmap="viridis")

In [ ]:
def sync_all_runs_to_drive():
    run_root = ROOT / "cloud_runs"
    synced = []
    for run_dir in sorted(run_root.glob("*")):
        if not run_dir.is_dir():
            continue
        dest = sync_run_to_drive(run_dir)
        if dest is not None:
            synced.append(str(dest))
    print(f"[drive] Synced {len(synced)} run folder(s)")
    return synced

# Run manually after a disconnect scares you, or to back up everything:
# sync_all_runs_to_drive()


## 9. Run history

Every `run_job(...)` call writes a `run_manifest.json` into a dedicated run directory.

In [ ]:
run_root = ROOT / "cloud_runs"
manifest_paths = sorted(run_root.glob("*/run_manifest.json"))
rows = []
for p in manifest_paths:
    data = json.loads(p.read_text())
    row = {
        "tag": data.get("tag"),
        "status": data.get("status"),
        "elapsed_s": data.get("elapsed_s"),
        "n_steps": data.get("n_steps"),
        "run_dir": str(Path(data["paths"]["run_dir"])),
    }
    row.update(data.get("telemetry", {}))
    rows.append(row)

history_df = pd.DataFrame(rows)
history_df.sort_values(["status", "tag"]) if not history_df.empty else history_df

## 10. Save job to disk

Writes the current in-memory `job` (from section 3) back to `jobs/notebook_session/`.


In [ ]:
session_jobs_dir = ROOT / "jobs" / "notebook_session"
saved_job = write_job_yaml(job, session_jobs_dir / "job.applied.yaml")
saved_torch = (session_jobs_dir / "torch_path.csv").read_text()
print("Saved job:", saved_job)
print("Torch CSV waypoints:", len(saved_torch.strip().splitlines()) - 1)


## 11. Recommended cloud/server workflow

For production-grade server use:

1. **Notebook for development / tuning** — edit parameters, run batches, inspect telemetry / VTK
2. **Saved tuned job YAMLs** — `jobs/tuned/*.yaml` as source of truth
3. **Headless batch scripts** — `python -m waam_twin.export ...` or loops calling `WAAMTwin.from_job(...)`
4. **Post-processing** — VTK bundles / sequences in ParaView or Python

This gives you essentially the same core functionality as local use, except the desktop GGUI viewer is replaced by notebook plots and VTK inspection.

## 12. Optional next upgrade

Possible future improvements:

- unify live view into `run_job(live_view=True, ...)`
- sweep result tables with sorting / filtering
- comparison against `reference` / `model_reference`
- job queue integration for remote compute nodes